 # QuickKart Big Data Analytics — Your First Databricks Lab

## MBA (Business Analytics) | Beginner-friendly | Databricks

**You are NOT here to become a programmer.**  
You are here to understand what the technology is doing, why a business uses it, and how an analyst turns raw data into a decision.

### Today's story

QuickKart is a 10-minute grocery/e-commerce company.

- **Ananya** — customer in Pune; order `QK-847291`; ₹2,340
- **Priya** — data analyst
- **Arjun** — data engineer
- **Meera** — analytics lead

We will follow QuickKart data through this journey:

`Orders → Spark/Databricks → Clean data → Filter → Group → Total → Sort → Business insight`

Then we will see where **Kafka, Cassandra, Delta Lake and SQOOP** fit.

---

## The one rule for this lab

You already know the analytics logic from Excel:

| Business action | Excel idea | Spark idea |
|---|---|---|
| Keep only some rows | Filter | `.filter()` |
| Put similar rows together | Pivot/group | `.groupBy()` |
| Calculate a number | SUM / AVERAGE | `.sum()` / `.avg()` |
| Rank results | Sort | `.orderBy()` |

**Only the tool changes. The business thinking does not.**

# Part 0 — I have never opened Databricks. What am I looking at?

On the Databricks home screen you uploaded, the left menu is the main navigation.

### The only areas you need today

**1. New**  
Think: **Create button**. Use it to create a notebook.

**2. Workspace**  
Think: **your course folder / Google Drive folder**. Your notebooks live here.

**3. Catalog**  
Think: **university library catalogue**. It tells you what tables/data are available.

**4. Compute**  
Think: **the computer/engine that does the work**. In some Databricks editions you choose a serverless environment or a compute resource.

**5. SQL Editor / Queries / Dashboards**  
Think: **the business-user side** — query data and make dashboards.

**6. Jobs & Pipelines**  
Think: **scheduled automation**. Instead of clicking Run every morning, Databricks can run a pipeline for you.

### Before importing this notebook

1. From the home screen click **Workspace**.
2. Open or create a folder for your class.
3. Choose **Import** and upload this `.ipynb` file.  
   *If your interface instead offers New → Notebook, you can create a notebook and import/copy content there.*
4. Open the notebook.
5. At the top, attach/select the available **compute/serverless** option if Databricks asks.
6. Run cells from top to bottom using the **▶ Run** button.

### Important beginner habit

Run **one cell at a time** the first time.

If a cell succeeds, move on.  
If a cell fails, read the **last line of the error first**.

# Part 1 — First contact with Spark

## Everyday twin: hotel reception desk

Databricks already gives us an object called `spark`.

Think of `spark` as the **reception desk**: one place from which we can access Spark features.

Run the next cell.

In [ ]:
print("Hello from Databricks!")
print("Spark version:", spark.version)
print("SparkSession is ready:", spark is not None)

### What should happen?

You should see something similar to:

```text
Hello from Databricks!
Spark version: ...
SparkSession is ready: True
```

**Business takeaway:** We have confirmed that the analytics engine is available.

**Question for the class:** Did we install Spark ourselves?  
**Answer:** No. Databricks provides the environment.

# Part 2 — Create QuickKart data

We will generate a reproducible QuickKart order dataset directly in Spark.



In [ ]:
from pyspark.sql import functions as F

# Number of synthetic QuickKart orders for the lab
N_ORDERS = 50_000

orders = (
    spark.range(1, N_ORDERS + 1)
    .withColumn("order_id", F.concat(F.lit("QK-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn(
        "city",
        F.when((F.col("id") % 5) == 0, "Pune")
         .when((F.col("id") % 5) == 1, "Mumbai")
         .when((F.col("id") % 5) == 2, "Delhi")
         .when((F.col("id") % 5) == 3, "Bengaluru")
         .otherwise("Hyderabad")
    )
    .withColumn(
        "category",
        F.when((F.col("id") % 4) == 0, "Grocery")
         .when((F.col("id") % 4) == 1, "Electronics")
         .when((F.col("id") % 4) == 2, "Home")
         .otherwise("Personal Care")
    )
    .withColumn("amount", (F.lit(250) + ((F.col("id") * 37) % 4750)).cast("double"))
    .withColumn(
        "status",
        F.when((F.col("id") % 20) == 0, "CANCELLED")
         .when((F.col("id") % 25) == 0, "REFUNDED")
         .otherwise("COMPLETED")
    )
    .withColumn("customer_id", F.concat(F.lit("C"), F.lpad(((F.col("id") % 8000) + 1).cast("string"), 5, "0")))
    .withColumn("order_hour", (F.col("id") % 24).cast("int"))
    .drop("id")
)

# Add Ananya's famous order explicitly
ananya = spark.createDataFrame(
    [("QK-847291", "Pune", "Grocery", 2340.0, "COMPLETED", "C08472", 21)],
    ["order_id", "city", "category", "amount", "status", "customer_id", "order_hour"]
)

orders = orders.unionByName(ananya)

print("QuickKart dataset created.")
print("Rows:", orders.count())

## Everyday twin: spreadsheet

A Spark **DataFrame** is just rows and columns.

That should feel familiar.

Run the next cell. In Databricks, `display()` gives a richer table than plain `show()`.

In [ ]:
display(orders.limit(20))

# Part 3 — Understand the shape of the data

Before analysis, Priya asks:

> "What exactly has Arjun given me?"

In Excel, you would look at the headers and perhaps the number of rows.

In Spark we can inspect the schema.

In [ ]:
orders.printSchema()

In [ ]:
print("Number of rows:", orders.count())
print("Number of columns:", len(orders.columns))
print("Columns:", orders.columns)

**Business takeaway:** Before asking a question, understand what data exists and what each field means.

This prevents a classic analytics mistake: answering a business question with the wrong column.

# Part 4 — Find Ananya's order

This is our first of the four verbs.

## VERB 1: FILTER

Meera asks:

> "Can you find Ananya's festive-sale order QK-847291?"

In Excel: turn on Filter and search for the order number.  
In Spark: `.filter()`.

In [ ]:
ananya_order = orders.filter(F.col("order_id") == "QK-847291")

display(ananya_order)

### Read the code as English

```text
FROM orders
KEEP the row
WHERE order_id = QK-847291
```

The syntax is not the lesson.

**The lesson is the business operation: FILTER.**

# Part 5 — A real management question

Meera now asks:

> "Ignore cancelled/refunded orders. Which city is generating the most completed-order revenue?"

We will answer this using the four verbs.

## Step 5A — FILTER completed orders

In [ ]:
completed_orders = orders.filter(F.col("status") == "COMPLETED")

print("All orders:", orders.count())
print("Completed orders:", completed_orders.count())

## Step 5B — GROUP orders by city

Every Mumbai order goes into one mental bucket, every Pune order into another, and so on.

## Step 5C — TOTAL the business metrics

For each city we calculate:

- orders
- revenue
- average order value (AOV)

In [ ]:
city_summary = (
    completed_orders
    .groupBy("city")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.avg("amount"), 2).alias("avg_order_value")
    )
)

display(city_summary)

## Step 5D — SORT from highest revenue to lowest

In [ ]:
city_ranked = city_summary.orderBy(F.col("revenue").desc())

display(city_ranked)

# Stop here and behave like an MBA, not a programmer

Do **not** ask, "What code did we type?"

Ask:

1. Which city is #1 by revenue?
2. Is the #1 city also #1 by average order value?
3. Would you allocate marketing budget purely based on revenue?
4. What additional information would you want before making that decision?

That is Business Analytics.

# Part 6 — Same question in SQL

A key lesson: Spark is not asking you to abandon what you know.

If you know SQL, you can ask the same question using SQL.

First create a temporary SQL view.

In [ ]:
orders.createOrReplaceTempView("quickkart_orders")

sql_result = spark.sql("""
SELECT
    city,
    COUNT(*) AS orders,
    ROUND(SUM(amount), 2) AS revenue,
    ROUND(AVG(amount), 2) AS avg_order_value
FROM quickkart_orders
WHERE status = 'COMPLETED'
GROUP BY city
ORDER BY revenue DESC
""")

display(sql_result)

### Compare the thinking

Spark DataFrame:

```text
filter → groupBy → sum/avg → orderBy
```

SQL:

```text
WHERE → GROUP BY → SUM/AVG → ORDER BY
```

Excel:

```text
Filter → Pivot/Group → Total → Sort
```

**Same four verbs. Different language.**

# Part 7 — Category performance

Meera asks:

> "Which product category contributes the most revenue?"

Try to predict the steps before running the cell.

1. Filter?
2. Group by what?
3. Total what?
4. Sort by what?

In [ ]:
category_summary = (
    orders
    .filter(F.col("status") == "COMPLETED")
    .groupBy("category")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.avg("amount"), 2).alias("avg_order_value")
    )
    .orderBy(F.col("revenue").desc())
)

display(category_summary)

# Part 8 — Peak-hour analysis

Operations asks:

> "At what hour do we receive the most completed orders?"

This is useful for:

- rider staffing
- customer support staffing
- compute capacity
- promotion timing

In [ ]:
hourly = (
    orders
    .filter(F.col("status") == "COMPLETED")
    .groupBy("order_hour")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("amount"), 2).alias("revenue")
    )
    .orderBy("order_hour")
)

display(hourly)

### Databricks visualisation

After the table appears, use the visualisation/chart option in the result area if your Databricks edition shows it.

Suggested chart:

- X-axis: `order_hour`
- Y-axis: `orders`
- Chart: line or bar

**Business question:** Which hours would you staff most heavily?

# Part 9 — Lazy execution: Spark plans first

## Everyday twin: recipe card

Spark is **lazy**.

It often records the steps you want before doing the expensive work.

Think:

> "Read the question first. Plan the route. Then execute."

Let's build a transformation without immediately asking for the final answer.

In [ ]:
planned_analysis = (
    orders
    .filter(F.col("status") == "COMPLETED")
    .filter(F.col("amount") >= 1000)
    .groupBy("city")
    .agg(F.sum("amount").alias("high_value_revenue"))
)

print("We have DEFINED the analysis.")
print("Now inspect Spark's execution plan:")
planned_analysis.explain()

Now trigger an **action** by asking Spark to return/display the answer.

In [ ]:
display(planned_analysis.orderBy(F.col("high_value_revenue").desc()))

### Twin

Defining transformations = writing the recipe.  
Calling an action such as `display()`, `count()` or `collect()` = telling the kitchen to serve.

For an MBA class, remember only this:

> **Spark can optimise because it sees the plan before executing it.**

# Part 10 — Cache: keep frequently used data on the hot rack

If we repeatedly analyse the same completed orders, Spark can cache them.

## Everyday twin

Instead of going back to the storeroom every time, keep frequently used ingredients on the kitchen counter/hot rack.

In [ ]:
completed_orders.cache()

# This action materialises the cache
completed_count = completed_orders.count()

print("Completed-order DataFrame cached.")
print("Rows cached:", completed_count)

Later, when finished:

In [ ]:
completed_orders.unpersist()
print("Cache released.")

# Part 11 — Delta Lake

## Everyday twin: bank transaction + Google Docs history

Delta Lake adds reliability to data-lake storage.

For business students, focus on two ideas:

1. **ACID reliability** — a transaction should complete correctly, not half-complete.
2. **Version history / time travel** — previous table versions can be inspected when the environment supports it.

We will try to create a managed Delta table.

> If your Databricks account does not permit table creation in the default location, this cell catches the error and the rest of the notebook still works.

In [ ]:
DELTA_TABLE = "quickkart_orders_delta"

try:
    spark.sql(f"DROP TABLE IF EXISTS {DELTA_TABLE}")
    (
        orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(DELTA_TABLE)
    )
    print(f"Delta table created: {DELTA_TABLE}")
    display(spark.table(DELTA_TABLE).limit(10))
except Exception as e:
    print("Delta table creation was not available in this workspace.")
    print("This can happen because of workspace/catalog permissions.")
    print("The Spark analysis sections of the lab are unaffected.")
    print("Error summary:", str(e)[:500])

If the table was created successfully, run this:

In [ ]:
try:
    detail = spark.sql(f"DESCRIBE DETAIL {DELTA_TABLE}")
    display(detail)
except Exception as e:
    print("Skipping DESCRIBE DETAIL because the managed Delta table is unavailable.")

# Part 12 — Where Kafka fits

So far we used a prepared DataFrame.

Real QuickKart does not wait until midnight to create one giant file.

Orders arrive continuously.

## Everyday twin: conveyor belt / newspaper subscription

Kafka is the **conveyor belt** carrying events.

```text
Customer app → Kafka → Databricks/Spark → analytics/storage
```

We can demonstrate the **streaming idea without needing an external Kafka server** by using Spark's built-in `rate` stream.

In [ ]:
# Stop an old demo stream if this cell was run previously
for q in spark.streams.active:
    if q.name == "quickkart_live_demo":
        q.stop()

live_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

live_orders = (
    live_source
    .withColumn("order_id", F.concat(F.lit("LIVE-"), F.col("value").cast("string")))
    .withColumn(
        "city",
        F.when((F.col("value") % 3) == 0, "Mumbai")
         .when((F.col("value") % 3) == 1, "Pune")
         .otherwise("Delhi")
    )
    .withColumn("amount", (F.lit(300) + ((F.col("value") * 43) % 2000)).cast("double"))
    .select("timestamp", "order_id", "city", "amount")
)

print("Streaming DataFrame created:", live_orders.isStreaming)

Now start the live stream and let it run for about 8 seconds.

In [ ]:
import time

live_query = (
    live_orders.writeStream
    .format("memory")
    .queryName("quickkart_live_demo")
    .outputMode("append")
    .start()
)

time.sleep(8)

live_query.processAllAvailable()

display(
    spark.sql("""
        SELECT *
        FROM quickkart_live_demo
        ORDER BY timestamp DESC
        LIMIT 30
    """)
)

Look at the timestamps.

These rows were generated **while the notebook was running**.

That is the mental jump from:

`data at rest`

to

`data in motion`.

Let's calculate a live business summary from what has arrived so far.

In [ ]:
display(
    spark.sql("""
        SELECT
            city,
            COUNT(*) AS orders_so_far,
            ROUND(SUM(amount), 2) AS revenue_so_far
        FROM quickkart_live_demo
        GROUP BY city
        ORDER BY revenue_so_far DESC
    """)
)

Stop the stream when the demonstration is over.

In [ ]:
if live_query.isActive:
    live_query.stop()
print("Live demo stream stopped.")

# Part 13 — What REAL Kafka code looks like

The previous section works without any external infrastructure.

A **real Kafka connection requires a Kafka broker/network/security configuration**.

Do not enable this cell unless Arjun (the data engineer) has given you:

- Kafka bootstrap server
- topic name
- credentials/security settings if required

The important part for MBA students is to **recognise** the pattern.

In [ ]:
RUN_REAL_KAFKA = False

if RUN_REAL_KAFKA:
    KAFKA_SERVER = "YOUR_SERVER:9092"
    KAFKA_TOPIC = "quickkart_orders"

    kafka_raw = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_SERVER)
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "latest")
        .load()
    )

    # Kafka's value is commonly bytes; convert it to readable text/JSON.
    kafka_text = kafka_raw.selectExpr(
        "CAST(value AS STRING) AS order_json",
        "timestamp",
        "topic",
        "partition",
        "offset"
    )

    display(kafka_text)
else:
    print("Real Kafka section is OFF — this is intentional for a self-contained classroom lab.")

### Read the real Kafka code as English

```text
readStream               → keep listening
format("kafka")           → source is Kafka
bootstrap.servers         → where Kafka lives
subscribe                 → which conveyor belt/topic
startingOffsets="latest"  → start with new events
load()                    → create the stream
```

You are not expected to memorise this.

# Part 14 — Where Cassandra fits

## Everyday twin: 24/7 store with identical branches

Cassandra is an **operational NoSQL database** designed for high availability and large distributed workloads.

QuickKart might use an operational store for questions such as:

> "What is the latest status of order QK-847291?"

That is a different workload from:

> "What was total revenue by city for the last six months?"

### Mental model

```text
Kafka       → data is MOVING
Delta       → analytical history
Cassandra   → fast operational serving
```

For a fully working Cassandra connection, you need:

- a Cassandra cluster
- network access from Databricks
- authentication details
- a compatible Spark Cassandra Connector installed on the compute

Therefore this notebook keeps the real connector **OFF by default**.

In [ ]:
RUN_REAL_CASSANDRA = False

if RUN_REAL_CASSANDRA:
    # Example pattern only.
    # Your data engineer must configure the Cassandra connector and connection settings.
    cassandra_orders = (
        spark.read
        .format("org.apache.spark.sql.cassandra")
        .options(
            keyspace="quickkart",
            table="order_status"
        )
        .load()
    )

    display(cassandra_orders.limit(20))
else:
    print("Real Cassandra section is OFF — external Cassandra infrastructure is required.")

# Part 15 — Where SQOOP fits

## Everyday twin: moving van

SQOOP belongs mainly to the traditional Hadoop world.

Its job was bulk relocation:

```text
MySQL / Oracle  ⇄  SQOOP  ⇄  Hadoop / HDFS
```

It is **not** the technology we would normally install inside a modern Databricks notebook.

A historical SQOOP command looked roughly like:

```bash
sqoop import   --connect jdbc:mysql://database/quickkart   --table customers   --target-dir /data/customers
```

For a modern Databricks architecture, you would more commonly use:

- JDBC for database reads/writes
- CDC (Change Data Capture)
- managed ingestion/connectors

So in class:

**SQOOP = recognise the old relocation pattern.**  
**JDBC/CDC = recognise the modern pattern.**

# Part 16 — Modern JDBC pattern

This cell is also OFF because a real database requires a URL, credentials and network connectivity.

The code is included so you can recognise what replaced many SQOOP-style workflows.

In [ ]:
RUN_REAL_JDBC = False

if RUN_REAL_JDBC:
    JDBC_URL = "jdbc:mysql://YOUR_DATABASE_HOST:3306/quickkart"

    customers = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", "customers")
        .option("user", "YOUR_USER")
        .option("password", "USE_A_SECRET_NOT_PLAINTEXT_IN_PRODUCTION")
        .load()
    )

    display(customers.limit(20))
else:
    print("Real JDBC section is OFF — no external database is needed for today's lab.")

# Part 17 — Put the whole architecture together

Follow Ananya's order.

```text
Ananya places order QK-847291
            │
            ▼
         KAFKA
   conveyor belt / event stream
            │
            ▼
   DATABRICKS + SPARK
  clean / transform / aggregate
       │             │
       ▼             ▼
   DELTA LAKE     CASSANDRA
   analytics      operational
   history        serving
       │
       ▼
     MEERA
business decision
```

And the older relocation story:

```text
Old SQL database → SQOOP → Hadoop/HDFS
```

Modern equivalent:

```text
SQL database → JDBC / CDC → Databricks / Delta
```

# Part 18 — Technology recognition test

Don't memorise definitions. Recognise the business need.

| Situation | Best mental match |
|---|---|
| 20,000 QuickKart events are arriving every second | **Kafka** |
| Run transformations and analytics at scale | **Spark** |
| Managed platform where Spark, notebooks, SQL and data engineering meet | **Databricks** |
| Reliable analytical storage with transaction/version capabilities | **Delta Lake** |
| Highly available operational NoSQL serving | **Cassandra** |
| Move bulk data between RDBMS and old Hadoop | **SQOOP** |
| Modern relational-database ingestion pattern | **JDBC / CDC** |

# Part 19 — Mini business challenge

You are now **Priya**.

Meera asks:

> "For completed QuickKart orders above ₹2,000, which city has the highest revenue?"

Before looking at any code, write the four steps:

1. FILTER: ______________________
2. GROUP: _______________________
3. TOTAL: _______________________
4. SORT: ________________________

In [ ]:
# Try it yourself first.
#
# Hint:
# orders
#   .filter(...)
#   .filter(...)
#   .groupBy(...)
#   .agg(...)
#   .orderBy(...)

high_value_city = (
    orders
    .filter(F.col("status") == "COMPLETED")
    .filter(F.col("amount") > 2000)
    .groupBy("city")
    .agg(
        F.count("*").alias("high_value_orders"),
        F.round(F.sum("amount"), 2).alias("high_value_revenue")
    )
    .orderBy(F.col("high_value_revenue").desc())
)

display(high_value_city)

# Part 20 — Change-one-thing exercises

These are deliberately small.

You are not writing a new program.  
You are changing **one business assumption** and observing the result.

### Exercise A
Change `amount > 2000` to `amount > 3000`.

**Question:** Does the city ranking change?

### Exercise B
Change `groupBy("city")` to `groupBy("category")`.

**Question:** Which category leads among high-value completed orders?

### Exercise C
Filter only `city == "Pune"`.

**Question:** Which category contributes the most completed revenue in Pune?

### Exercise D
Use `order_hour` as the group.

**Question:** At what hour is high-value revenue strongest?

# Part 21 — MBA discussion: what would you tell the business?

Choose ONE output from today's notebook and answer:

### Observation
What happened?

### Interpretation
Why might it be happening?

### Decision
What should QuickKart do?

### Risk
What additional data would you want before acting?

This separation matters.

**Data does not make the decision. Managers do.**

# Part 22 — Clean-up

This keeps the classroom workspace tidy.

Run it only when you are finished.

In [ ]:
# Stop any stream that might still be running
for q in spark.streams.active:
    if q.name == "quickkart_live_demo":
        q.stop()

# Remove cache if any
try:
    completed_orders.unpersist()
except Exception:
    pass

print("Lab clean-up complete.")

# Final recap — Seven sentences to remember

1. **DataFrame** = spreadsheet at scale.
2. **Spark** = engine that processes large/distributed data.
3. **Databricks** = platform built around data engineering, analytics and Spark capabilities.
4. **Kafka** = conveyor belt for data in motion.
5. **Cassandra** = highly available operational NoSQL store.
6. **Delta Lake** = reliable analytical storage.
7. **SQOOP** = older moving van between relational databases and Hadoop.

And underneath nearly everything we did:

> **FILTER → GROUP → TOTAL → SORT**

You already knew the analytics.  
Today you changed where it runs.